## What is LangGraph?

LangGraph is an open-source framework and orchestration engine created by LangChain for building stateful, multi-actor AI agents. 

It allow to define AI workflows using a graph-based architecture—where nodes represent specific tasks (like calling an LLM or running a tool) and edges manage complex logic like loops, conditional branching, and parallel processing. 

## when to use LangGraph vs LangChain

- Use Langchain for simple LLM applications that require a single prompt and response such as basic chatbots and Q&A systems.
- It is ideal for building RAG pipelines and one-shot question-answering applications.
- It is an excellent choice for rapid prototyping and building MVPs quickly.

- Use LangGraph when your application requires stateful workflows that maintain context across multiple steps.
- It is designed for workflows that need loops, retries, or self-correcting behavior.
- It is the right choice when branching or conditional execution is required based on decisions.
- It supports human-in-the-loop workflows where approvals or reviews are part of the process.
- It is best suited for production-grade AI agents, multi-agent systems, and long-running workflows.

## Use cases
Here are the **real-world use cases of LangGraph** in clear, concise sentences:

1. **Research Agents:** Build AI agents that can search for information, evaluate results, and iterate until they find the best answer.
2. **Code Review Agents:** Create agents that analyze code, suggest improvements, recheck changes, and validate the final output.
3. **Approval Workflows:** Design workflows that include human approvals and reviews, even if the process spans multiple days.
4. **AI and Multi-Agent Systems:** Develop intelligent AI agents that collaborate with other agents to solve complex tasks.
5. **Long-Running Workflows:** Manage stateful processes that require multiple steps, decision-making, retries, and continuous execution over time.

## Key Concepts

## State: 

A shared data structure (like a whiteboard) that acts as the application's memory. Every node can read from it and write updates back to it as the workflow runs.

```python
from typing import TypedDict

class State(TypedDict):
    question: str
    answer: str
```

## Nodes: 

Plain Python or TypeScript functions that perform specific operations. They take the current state, process it, and return updated values. 

```python
def chatbot(state: State):
    return {
        "answer": "Hello!"
    }
```

## Edges:

Connections that determine the flow of execution. They can be static or conditional (e.g., "if the data is invalid, return to the editing node")

## why handoffs in LangGprah

Handoffs in LangGraph allow one agent to transfer control to another agent when a different agent is better suited to handle the next part of the task.

### Why are handoffs needed?
- Specialized agents – Different agents can have different expertise (e.g., a Research Agent, Coding Agent, and Review Agent).
- Task delegation – An agent can pass a task to another agent that is better equipped to complete it.
- Modular workflows – Each agent focuses on a specific responsibility, making the system easier to build and maintain.
- Improved accuracy – The most appropriate agent handles each step, leading to better results.
- Scalable multi-agent systems – New agents can be added without redesigning the entire workflow.

## Direct Edge VS Conditional Edge

### Direct Edge: 

A direct edge connects one node to another in a fixed sequence. After the current node finishes execution, the graph always moves to the same next node without checking any conditions. It is best for simple, linear workflows.

```python
graph.add_edge("A", "B")
```

### Conditional Edge: 

A conditional edge evaluates the current state or the output of a node to decide which node should execute next. This enables branching, routing, retries, tool selection, and other dynamic behaviors, making it ideal for AI agents and complex workflows.

```python
graph.add_conditional_edges("A", router, {
    "search": "search",
    "answer": "answer"
})
```

## Basic Routing VS Literal Routing 

### Basic Routing

Basic routing uses a routing function with standard strings to determine the next node. The function contains custom logic (such as if-else conditions) to decide where the graph should go next.

```python
def router(state):
    if state["need_search"]:
        return "search"
    return "generate"
```

### Literal Routing

Literal routing uses Python's Literal type to restrict the router's possible outputs to a predefined set of values. This makes the routing logic more type-safe, easier to validate, and improves IDE support.

```python
from typing import Literal

def router(state) -> Literal["search", "generate"]:
    if state["need_search"]:
        return "search"
    return "generate"
```

## Multipath Routing

**Multipath routing** allows a node to route execution to **multiple next nodes simultaneously** instead of just one. It is useful when multiple tasks can run independently in parallel.

### Example

```python
def router(state):
    return ["search", "calculator"]

graph.add_conditional_edges(
    "planner",
    router,
    {
        "search": "search",
        "calculator": "calculator"
    }
)
```

Here are some practical **real-world use cases** for **Multipath Routing** in LangGraph:

### 1. Research Agent (Most Common)

**Scenario:** A user asks, *"What are the latest AI trends in healthcare?"*

The planner routes the query to multiple sources in parallel:

* 🌐 Web Search
* 📄 Internal Knowledge Base
* 📚 Research Papers

```text
           User Query
                │
            Planner
      ┌─────────┼─────────┐
      │         │         │
 Web Search  Knowledge  Research Papers
      │         │         │
      └─────────┼─────────┘
                │
         Merge Results
                │
        Generate Answer
```

**Benefit:** Faster retrieval and more comprehensive answers.

---

### 2. Financial Analysis Agent

**Scenario:** A user asks, *"Should I invest in Company X?"*

The agent gathers information from multiple sources:

* Latest News
* Financial Statements
* Stock Market Data

```text
             Investment Query
                    │
                Planner
        ┌───────────┼───────────┐
        │           │           │
     News API   Financials   Stock Data
        │           │           │
        └───────────┼───────────┘
                    │
             Investment Report
```

**Benefit:** Produces a more accurate recommendation using multiple data sources.

---

### 3. Multi-Tool AI Assistant

**Scenario:** User asks, *"Schedule a meeting tomorrow and email the attendees."*

The planner invokes multiple tools:

* Calendar Tool
* Email Tool
* Contacts Lookup

```text
             User Request
                  │
              Planner
      ┌───────────┼───────────┐
      │           │           │
 Calendar     Contacts      Email
      │           │           │
      └───────────┼───────────┘
                  │
          Confirmation
```

**Benefit:** Independent tasks run in parallel, making the assistant more responsive.

---
